# QUBO / Ising Combinatorial Optimisation

Phase 0 scaffold for **binary quadratic** problems:

| Solver | When |
|--------|------|
| `chain_qubo_exact_dp` | 1D chain topology, **exact** O(n) |
| `qubo_brute_force` | n ≤ ~20 |
| `qubo_simulated_annealing` | general dense Q |
| `qubo_mps_dmrg_chain` | chain + MPS product-state sweeps |

Portfolio subset selection is encoded as a QUBO via `portfolio_selection_qubo`.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "../.."))
Pkg.instantiate()

using MPSFast
using Random, LinearAlgebra, Printf

  Activating project at `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl`


## 1. Path MaxCut (chain — exact DP)

In [2]:
prob = maxcut_path_qubo(20)
chain = chain_qubo_from_dense(prob)
rows = qubo_solver_compare(chain)

println(@sprintf("%-12s %10s", "method", "energy"))
for r in rows
    @printf "%-12s %10.4f\n" string(r.method) r.energy
end

method           energy
chain_dp        19.0000
mps_dmrg        19.0000
local           19.0000
anneal          19.0000


## 2. Random spin glass (chain)

In [3]:
rng = MersenneTwister(42)
chain = ChainQUBOProblem(randn(rng, 30), randn(rng, 29))
exact = chain_qubo_exact_dp(chain)
mps   = qubo_mps_dmrg_chain(chain; n_sweeps=40, rng=rng)
sa    = qubo_simulated_annealing(chain; rng=rng)

@printf "exact DP   E = %.6f\n" exact.energy
@printf "MPS sweep  E = %.6f\n" mps.energy
@printf "anneal     E = %.6f\n" sa.energy

exact DP   E = -13.956548
MPS sweep  E = -13.956548
anneal     E = -13.956548


## 3. Portfolio subset selection (dense QUBO)

In [4]:
rng = MersenneTwister(7)
n = 10
μ = randn(rng, n)
A = randn(rng, n, n)
Σ = A' * A + 2.0I
target_k = 5

exact = optimize_portfolio_exact_k(μ, Σ; k=target_k, λ=0.5)
prob = portfolio_selection_qubo(μ, Σ; λ=0.5, target_k=target_k, penalty=500.0)
brute = qubo_brute_force(prob)
sa    = qubo_simulated_annealing(prob; rng=rng, n_steps=100_000)
loc   = qubo_local_search(prob; rng=rng)

println("Exact-k selected (k=$target_k): ", findall(==(1), exact.x))
println("QUBO brute selected: ", findall(==(1), brute.x))
@printf "exact E = %.4f   brute E = %.4f   SA E = %.4f\n" exact.energy brute.energy sa.energy

Exact-k selected (k=5): [1, 2, 3, 4, 6]
QUBO brute selected: [1, 3, 4, 7, 8]
exact E = 8.0223   brute E = 15.5879   SA E = 19.2971


## 4. Number partitioning

In [5]:
s = [3, 7, 11, 13, 17, 19]
prob = number_partition_qubo(s)
sol = qubo_brute_force(prob)
println("partition: ", sol.x)
@printf "imbalance energy = %.1f\n" sol.energy

partition: [0, 0, 0, 0, 0, 1]
imbalance energy = -342.0


## 5. Generative + QUBO portfolio loop

**Pipeline:** train factor MPS → sample paths → multivariate scenarios → estimate `μ̂, Σ̂` → **exact-k Markowitz** (`optimize_portfolio_exact_k`) → OOS eval vs Gaussian MC baseline.

The soft QUBO penalty (`portfolio_selection_qubo`) remains for annealing/MPS solvers; the demo uses exact subset enumeration so `target_k` is enforced.

In [6]:
demo = demo_generative_portfolio_loop(;
    n_assets=8, n_train=3_000, n_oos=800, target_k=4,
    M=12, mps_epochs=35, λ=0.5,
    rng=MersenneTwister(2025))

m = demo.cmp.mps
r = demo.cmp.ref
target_k = 4

println("── Training selections (target_k=$target_k) ──")
println("  MPS selected k=$(m.train.n_selected) : ", m.train.selected)
println("  Gaussian selected k=$(r.train.n_selected) : ", r.train.selected)

println("\n── OOS equal-weight (fresh Gaussian scenarios) ──")
@printf "  MPS→QUBO  mean=%.5f  std=%.5f  sharpe=%.3f\n" m.oos.mean m.oos.std m.oos.sharpe
@printf "  MC→QUBO   mean=%.5f  std=%.5f  sharpe=%.3f\n" r.oos.mean r.oos.std r.oos.sharpe

println("\n── Estimated vs true first asset ──")
@printf "  μ̂[1]=%.5f  μ_true[1]=%.5f\n" m.train.μ_hat[1] demo.μ_true[1]

── Training selections (target_k=4) ──
  MPS selected k=4 : [1, 4, 6, 7]
  Gaussian selected k=4 : [1, 3, 4, 7]

── OOS equal-weight (fresh Gaussian scenarios) ──
  MPS→QUBO  mean=0.00440  std=0.51366  sharpe=0.009
  MC→QUBO   mean=0.00697  std=0.29601  sharpe=0.024

── Estimated vs true first asset ──
  μ̂[1]=-0.01461  μ_true[1]=-0.00126
